# Elektriciteitstarieven: vast (DATS24) vs dynamisch (EPEX)

Vergelijking van de totale elektriciteitskost voor twee scenario's:

| Scenario | Beschrijving |
|---|---|
| **A** | Geen batterij — vast DATS24-tarief vs dynamisch EPEX-tarief |
| **B** | Met geoptimaliseerde batterij — dynamisch EPEX-tarief (minimale kost) |

**Netkosten (Fluvius)** zijn identiek in beide scenario's.

> *Tariefwaarden zijn representatieve 2025-schattingen.*
> *Pas de configuratiecellen aan indien je over exacte contractwaarden beschikt.*

In [ ]:
import sys
from pathlib import Path

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import ipywidgets as widgets
from IPython.display import display
from scipy.optimize import linprog
from scipy.sparse import lil_matrix

from scripts.config import FINAL_DIR, INTERMEDIATE_DIR
from scripts import epex
import warnings; warnings.filterwarnings('ignore')

## 1. Tariefconfiguratie

### DATS24 leveringsprijs (exclusief netkosten, exclusief BTW)

Bronnen: VREG prijzendatabank, DareToCompare.be (april 2026).
Pas de waarden aan op basis van je eigen contract.

In [ ]:
# ── DATS24 vast tarief ──────────────────────────────────────────────────
DATS24 = {
    'dag_kwh':       0.1479,   # €/kWh leveringsprijs dag (7:00-22:00 weekdagen)
    'nacht_kwh':     0.1240,   # €/kWh leveringsprijs nacht
    'injectie_kwh':  0.0655,   # €/kWh injectievergoeding (VREG apr-2026)
    'maandkost':     2.24,     # €/maand vaste leveringskost
    'btw':           0.06,     # BTW-factor (6%)
}

# ── Dynamisch tarief (EPEX SPOT + opslag) ────────────────────────────────
DYNAMISCH = {
    'opslag_kwh':      0.0140,  # €/kWh vaste opslag boven EPEX-prijs
    'injectie_korting':0.0050,  # €/kWh korting op EPEX bij teruglevering
    'maandkost':       2.24,    # €/maand vaste leveringskost
    'btw':             0.06,
}

# ── Fluvius netkosten 2025 (Vilvoorde) ───────────────────────────────────
# Identiek voor vast én dynamisch tarief.
NETKOSTEN = {
    # Capaciteitstarief (maandpiek in kW, minimum 2.5 kW)
    'cap_eur_kw_jaar':  44.60,   # €/kW/jaar
    'cap_min_kw':        2.5,    # kW minimum
    # Distributienetkosten afname
    'dist_dag_kwh':     0.0295,  # €/kWh dag
    'dist_nacht_kwh':   0.0138,  # €/kWh nacht
    # Transmissie (flat)
    'trans_kwh':        0.0042,  # €/kWh
    # Injectie netkosten (afname-richting bij teruglevering)
    'inj_net_kwh':      0.0016,  # €/kWh
    # Vaste meterkost
    'meter_mnd':        3.75,    # €/maand
    # Heffingen & bijdragen (REG, CREG, DVO, Energiefonds)
    'heffingen_kwh':    0.0218,  # €/kWh variabel
    'heffingen_mnd':    1.85,    # €/maand vast
    'btw':              0.06,
}

# ── Batterijparameters ────────────────────────────────────────────────────
BAT = {
    'cap_kwh':   6.5,   # Bruikbare capaciteit (kWh)
    'max_kw':    3.0,   # Max laad-/ontlaadvermogen (kW)
    'eta_c':     0.95,  # Laadrendement
    'eta_d':     0.95,  # Ontlaadrendement
    'soc_min':   0.05,  # Minimum SOC (fractie)
    'soc_max':   0.95,  # Maximum SOC (fractie)
    'soc_start': 0.30,  # Beginstand per dag (fractie)
}

for naam, d in [('DATS24', DATS24), ('DYNAMISCH', DYNAMISCH), ('NETKOSTEN', NETKOSTEN), ('BATTERIJ', BAT)]:
    print(f'{naam}: {d}')

## 2. Data laden

### 2a. Kwartierdata (Fluvius basis)

In [ ]:
# Laad overall.csv
df = pd.read_csv(ROOT / 'data/Final/overall.csv', parse_dates=['kwartier'])

# Afgeleide kolommen
df['datum'] = pd.to_datetime(df['kwartier']).dt.date
df['uur']   = pd.to_datetime(df['kwartier']).dt.floor('h')
df['kwartier_van_dag'] = (
    pd.to_datetime(df['kwartier']).dt.hour * 4
    + pd.to_datetime(df['kwartier']).dt.minute // 15
)

# Batterijflows per kwartier (iLumen uurwaarden /4)
df['bat_charge_q']    = df['bat_geladen_kwh'].fillna(0)  / 4
df['bat_discharge_q'] = df['bat_ontladen_kwh'].fillna(0) / 4

# Basisbelasting zonder batterij:
#   net_load = afname - injectie = huis + ev + bat_charge - bat_discharge - zon
#   base_load_no_bat = net_load - bat_charge + bat_discharge
df['base_load_kwh'] = (
    df['afname_kwh'] - df['injectie_kwh']
    - df['bat_charge_q'] + df['bat_discharge_q']
)

print(f'Kwartieren  : {len(df)}')
print(f'Periode     : {df["kwartier"].min().date()} -> {df["kwartier"].max().date()}')
print(f'Basis afname: {df["base_load_kwh"].clip(lower=0).sum():.1f} kWh')
print(f'Basis inject: {(-df["base_load_kwh"]).clip(lower=0).sum():.1f} kWh')

### 2b. EPEX SPOT-prijzen ophalen

Dag-vooruit-prijzen voor België via de ENTSO-E Transparency API.

**Eenmalige setup** (API-sleutel opslaan):
```
python -c "import keyring; keyring.set_password('V1Eindwerk', 'entso_api_key', '<jouw-sleutel>')"
```
Sleutel aanvragen: https://transparency.entsoe.eu/  (gratis registratie)

De opgehaalde prijzen worden gecached in `data/intermediate results/epex_be.csv`.

In [ ]:
ts_min = str(df['datum'].min())
ts_max = str(df['datum'].max())
print(f'Perioden waarvoor EPEX nodig: {ts_min} -> {ts_max}')

df_epex = epex.load()
if df_epex.empty:
    print('Geen gecachede EPEX-prijzen gevonden. Ophalen...')
    df_epex = epex.fetch_and_save(ts_min, ts_max)
else:
    print(f'Gecached: {df_epex.index.min()} -> {df_epex.index.max()}')
    ontbreekt = len(df_epex) < (pd.Timestamp(ts_max) - pd.Timestamp(ts_min)).days * 20
    if ontbreekt:
        print('Vernieuwen van EPEX-data...')
        df_epex = epex.fetch_and_save(ts_min, ts_max)

print(f'EPEX uren: {len(df_epex)}, bereik: {df_epex.index.min()} -> {df_epex.index.max()}')
if not df_epex.empty:
    print(f'Gem. prijs: {df_epex["price_eur_mwh"].mean():.1f} €/MWh')
    print(f'Min prijs : {df_epex["price_eur_mwh"].min():.1f} €/MWh')
    print(f'Max prijs : {df_epex["price_eur_mwh"].max():.1f} €/MWh')

In [ ]:
# Koppel EPEX-prijs aan kwartierdata (uur -> kwartier)
# df['uur'] is tz-naive; df_epex.index is tz-aware (Europe/Brussels)
# Maak een tz-naive hulpindex voor de join
if not df_epex.empty:
    epex_naive = df_epex.copy()
    epex_naive.index = epex_naive.index.tz_localize(None)
    df['epex_eur_mwh'] = df['uur'].map(epex_naive['price_eur_mwh'])
    print(f'Kwartieren met EPEX-prijs: {df["epex_eur_mwh"].notna().sum()} / {len(df)}')
    print(f'Kwartieren zonder prijs  : {df["epex_eur_mwh"].isna().sum()}')
else:
    print('WAARSCHUWING: geen EPEX-prijzen beschikbaar.')
    print('Dynamisch-tariefberekening wordt overgeslagen.')
    df['epex_eur_mwh'] = np.nan

## 3. Kostenberekenisfuncties

### 3a. Fluvius netkosten (identiek voor alle scenario's)

De **capaciteitstarief** is gebaseerd op de **hoogste 15-minuten-gemiddelde**
netafname per maand (minimaal 2,5 kW).

In [ ]:
def netkosten_kwartier(df_in: pd.DataFrame) -> pd.Series:
    """
    Bereken de variabele netkosten (excl. BTW) per kwartier.
    Formule: (dist + trans + heffingen) per kWh afname + injectie-netkosten.
    """
    afname   = df_in['afname_kwh']
    injectie = df_in['injectie_kwh']
    tarief   = df_in['tarief']
    dist = np.where(tarief == 'dag', NETKOSTEN['dist_dag_kwh'], NETKOSTEN['dist_nacht_kwh'])
    var = afname * (dist + NETKOSTEN['trans_kwh'] + NETKOSTEN['heffingen_kwh'])
    var -= injectie * NETKOSTEN['inj_net_kwh']
    return pd.Series(var, index=df_in.index)


def capaciteitstarief_maand(afname_kwh: pd.Series,
                            kwartier_ts: pd.Series) -> pd.DataFrame:
    """
    Bereken het capaciteitstarief per maand op basis van de maandpiek (kW).

    Piek = maximum 15-min-gemiddeld vermogen [kW] = max(afname_kwh / 0.25).
    Minimumpiek = 2.5 kW.
    Jaarkosten = gem. maandpiek * cap_eur_kw_jaar; maandkost = 1/12 hiervan.
    """
    s = pd.Series(afname_kwh.values / 0.25,  # kW
                  index=pd.to_datetime(kwartier_ts))
    maandpiek = s.resample('MS').max().clip(lower=NETKOSTEN['cap_min_kw'])
    maandkost = maandpiek * NETKOSTEN['cap_eur_kw_jaar'] / 12
    return pd.DataFrame({'piek_kw': maandpiek, 'cap_kost': maandkost})


def vaste_netkosten_maand(kwartier_ts: pd.Series) -> pd.DataFrame:
    """
    Vaste maandelijkse netkosten (meter + heffingen vast), excl. BTW.
    """
    maanden = pd.to_datetime(kwartier_ts).to_period('M').drop_duplicates().to_timestamp()
    vaste = NETKOSTEN['meter_mnd'] + NETKOSTEN['heffingen_mnd']
    return pd.DataFrame({'vaste_nk': vaste}, index=maanden)

## 4. Scenario A — Zonder batterij

De **basisbelasting** (zonder batterij) is de netto gridflow die er zou zijn
als de batterij niet aanwezig was:

```
base_load = afname - injectie - bat_discharge + bat_charge
```

Positief = netto afname, negatief = netto injectie (overproductie zon).

| Kolom | Beschrijving |
|---|---|
| `afname_nb` | Netafname zonder batterij (kWh/kwartier) |
| `injectie_nb` | Injectie zonder batterij (kWh/kwartier) |

In [ ]:
# Basisscenario: afname/injectie zonder batterij
df['afname_nb']   = df['base_load_kwh'].clip(lower=0)
df['injectie_nb'] = (-df['base_load_kwh']).clip(lower=0)

# Hulpkolom: tarief op basis van kwartier-tijdstip (zelfde als Fluvius-tarief)
# (tarief 'dag' of 'nacht' blijft ongewijzigd)

# ── Energiekosten zonder batterij ────────────────────────────────────────
def energiekosten_vast(afname_kwh, injectie_kwh, tarief):
    """Energiekost (excl. netkosten, excl. BTW) per kwartier — vast DATS24-tarief."""
    prijs = np.where(tarief == 'dag', DATS24['dag_kwh'], DATS24['nacht_kwh'])
    return afname_kwh * prijs - injectie_kwh * DATS24['injectie_kwh']

def energiekosten_dynamisch(afname_kwh, injectie_kwh, epex_eur_mwh):
    """Energiekost (excl. netkosten, excl. BTW) per kwartier — dynamisch tarief."""
    prijs_kwh   = epex_eur_mwh / 1000 + DYNAMISCH['opslag_kwh']
    verkoop_kwh = epex_eur_mwh / 1000 - DYNAMISCH['injectie_korting']
    return afname_kwh * prijs_kwh - injectie_kwh * verkoop_kwh


# Kosten per kwartier — Scenario A (zonder batterij)
nb_mask = df['epex_eur_mwh'].notna()  # alleen kwartieren met EPEX-prijs

df['nk_kwartier'] = netkosten_kwartier(df.assign(
    afname_kwh=df['afname_nb'], injectie_kwh=df['injectie_nb']
))

df['ek_vast_nb'] = energiekosten_vast(
    df['afname_nb'], df['injectie_nb'], df['tarief']
)
df['ek_dyn_nb'] = np.where(
    nb_mask,
    energiekosten_dynamisch(
        df['afname_nb'], df['injectie_nb'], df['epex_eur_mwh'].fillna(0)
    ),
    np.nan,
)

print('Totalen scenario A (zonder batterij):')
print(f'  Energiekost vast       : {df["ek_vast_nb"].sum():>8.2f} €')
print(f'  Energiekost dynamisch  : {df["ek_dyn_nb"].sum():>8.2f} € (waar EPEX beschikbaar)')
print(f'  Variabele netkosten    : {df["nk_kwartier"].sum():>8.2f} €')

### 4a. Maandoverzicht — Scenario A

In [ ]:
df['maand'] = pd.to_datetime(df['kwartier']).dt.to_period('M')

# Capaciteitstarief per maand (zonder batterij = piek van base_load)
cap_nb = capaciteitstarief_maand(df['afname_nb'], df['kwartier'])

maand_a = df.groupby('maand').agg(
    ek_vast  =('ek_vast_nb',  'sum'),
    ek_dyn   =('ek_dyn_nb',   'sum'),
    nk_var   =('nk_kwartier', 'sum'),
    afname   =('afname_nb',   'sum'),
    injectie =('injectie_nb', 'sum'),
).round(2)

# Voeg capaciteitstarief en vaste kosten toe
cap_nb.index = cap_nb.index.to_period('M')
maand_a = maand_a.join(cap_nb[['piek_kw','cap_kost']], how='left')
maand_a['cap_kost'] = maand_a['cap_kost'].fillna(0)
maand_a['vast_mnd']  = NETKOSTEN['meter_mnd'] + NETKOSTEN['heffingen_mnd']
maand_a['lev_mnd']   = DATS24['maandkost']

btw = 1 + NETKOSTEN['btw']
maand_a['totaal_vast'] = (
    (maand_a['ek_vast'] + maand_a['lev_mnd']) * (1 + DATS24['btw'])
    + (maand_a['nk_var'] + maand_a['cap_kost'] + maand_a['vast_mnd']) * btw
)
maand_a['totaal_dyn'] = (
    (maand_a['ek_dyn'] + DYNAMISCH['maandkost']) * (1 + DYNAMISCH['btw'])
    + (maand_a['nk_var'] + maand_a['cap_kost'] + maand_a['vast_mnd']) * btw
)
maand_a['verschil'] = maand_a['totaal_dyn'] - maand_a['totaal_vast']
maand_a['verschil_pct'] = (maand_a['verschil'] / maand_a['totaal_vast'] * 100).round(1)

print('\nMAANDOVERZICHT — SCENARIO A (zonder batterij)\n')
kop = f'{"Maand":<9} {"Piek kW":>8} {"Vast €":>9} {"Dyn €":>9} {"Verschil €":>12} {"Verschil %":>11}'
print(kop)
print('-' * 62)
for m, r in maand_a.iterrows():
    print(f'{str(m):<9} {r["piek_kw"]:>8.2f} {r["totaal_vast"]:>9.2f}'
          f' {r["totaal_dyn"]:>9.2f} {r["verschil"]:>+12.2f} {r["verschil_pct"]:>+10.1f}%')
print('-' * 62)
print(f'{"TOTAAL":<9} {"":>8} {maand_a["totaal_vast"].sum():>9.2f}'
      f' {maand_a["totaal_dyn"].sum():>9.2f}'
      f' {maand_a["verschil"].sum():>+12.2f}'
      f' {maand_a["verschil"].sum()/maand_a["totaal_vast"].sum()*100:>+10.1f}%')

In [ ]:
# Staafdiagram: maandelijkse kost vast vs dynamisch (zonder batterij)
fig_a = go.Figure()
maanden_str = [str(m) for m in maand_a.index]

fig_a.add_trace(go.Bar(
    name='Vast (DATS24)',
    x=maanden_str, y=maand_a['totaal_vast'],
    marker_color='steelblue',
))
fig_a.add_trace(go.Bar(
    name='Dynamisch (EPEX)',
    x=maanden_str, y=maand_a['totaal_dyn'],
    marker_color='darkorange',
))
fig_a.add_trace(go.Scatter(
    name='Verschil (dyn-vast)',
    x=maanden_str, y=maand_a['verschil'],
    mode='lines+markers', line=dict(color='crimson', dash='dash'),
    yaxis='y2',
))

fig_a.update_layout(
    title='Maandelijkse elektriciteitskost: vast vs dynamisch (zonder batterij)',
    yaxis=dict(title='Kost incl. BTW (€/maand)'),
    yaxis2=dict(title='Verschil (€)', overlaying='y', side='right',
                zeroline=True, zerolinecolor='gray'),
    barmode='group',
    legend=dict(orientation='h', y=1.1),
    height=480,
)
fig_a.show()

## 5. Scenario B — Batterij LP-optimalisatie

De stationaire thuisbatterij wordt per dag optimaal ingezet om de
energiekost onder het **dynamisch tarief** te minimaliseren.

**LP-formulering per dag (T = 96 kwartieren):**

```
Minimaliseer:  Σ_t [ import_t * koop_t - export_t * verkoop_t ]

Subject to:
  (1) SOC[t] = SOC[t-1] + charge[t]*η_c - discharge[t]/η_d
  (2) import[t] - export[t] = base_load[t] + charge[t] - discharge[t]
  (3) SOC_min ≤ SOC[t] ≤ SOC_max
  (4) 0 ≤ charge[t], discharge[t] ≤ P_max * Δt
  (5) import[t], export[t] ≥ 0
```

**Variabelen per dag:** charge(96) + discharge(96) + soc(96) + import(96) + export(96) = 480

De SOC aan het einde van elke dag wordt overgedragen naar de volgende dag.

In [ ]:
def optimize_bat_dag(base_load: np.ndarray,
                     koop_prijs: np.ndarray,
                     verkoop_prijs: np.ndarray,
                     soc_start_kwh: float) -> dict:
    """
    LP-optimalisatie van batterijdispatch voor 1 dag (96 kwartieren).

    Parameters
    ----------
    base_load      : kWh per kwartier (positief = afname, negatief = injectie)
    koop_prijs     : €/kWh per kwartier (inkoopprijs stroom)
    verkoop_prijs  : €/kWh per kwartier (verkoopprijs bij teruglevering)
    soc_start_kwh  : SOC in kWh aan begin van dag

    Returns
    -------
    dict met charge, discharge, soc, grid_import, grid_export (allemaal kWh-arrays)
    en 'success' bool en 'soc_eind' float.
    """
    T   = len(base_load)
    dt  = 0.25  # uur per kwartier
    cap = BAT['cap_kwh']
    pmax_q  = BAT['max_kw'] * dt  # max kWh per kwartier
    soc_min = BAT['soc_min'] * cap
    soc_max = BAT['soc_max'] * cap
    ec = BAT['eta_c']
    ed = BAT['eta_d']

    # Variabelen: [charge(T), discharge(T), soc(T), import(T), export(T)]
    Nc, Nd, Ns, Ni, Ne = 0, T, 2*T, 3*T, 4*T
    n_vars = 5 * T

    # Doelfunctie: minimaliseer import*koop - export*verkoop
    c_obj = np.zeros(n_vars)
    c_obj[Ni:Ni+T] =  koop_prijs
    c_obj[Ne:Ne+T] = -verkoop_prijs

    # Gelijkheidsconstraints: A_eq * x = b_eq
    # (1) SOC dynamica: soc[t] - charge[t]*ec + discharge[t]/ed - soc[t-1] = soc_start if t==0
    # (2) Netverdeling: import[t] - export[t] - charge[t] + discharge[t] = base_load[t]
    A_eq = np.zeros((2*T, n_vars))
    b_eq = np.zeros(2*T)

    for t in range(T):
        # SOC-dynamica
        A_eq[t, Ns+t]  =  1.0
        A_eq[t, Nc+t]  = -ec
        A_eq[t, Nd+t]  =  1.0/ed
        if t > 0:
            A_eq[t, Ns+t-1] = -1.0
        b_eq[t] = soc_start if t == 0 else 0.0
        # Netverdeling
        A_eq[T+t, Ni+t] =  1.0
        A_eq[T+t, Ne+t] = -1.0
        A_eq[T+t, Nc+t] = -1.0
        A_eq[T+t, Nd+t] =  1.0
        b_eq[T+t] = base_load[t]

    bounds = (
        [(0, pmax_q)]  * T +   # charge
        [(0, pmax_q)]  * T +   # discharge
        [(soc_min, soc_max)] * T +  # soc
        [(0, None)] * T +      # import
        [(0, None)] * T        # export
    )

    res = linprog(c_obj, A_eq=A_eq, b_eq=b_eq, bounds=bounds, method='highs')

    if res.success:
        return {
            'success':     True,
            'charge':      res.x[Nc:Nc+T],
            'discharge':   res.x[Nd:Nd+T],
            'soc':         res.x[Ns:Ns+T],
            'grid_import': res.x[Ni:Ni+T],
            'grid_export': res.x[Ne:Ne+T],
            'soc_eind':    res.x[Ns+T-1],
            'cost':        res.fun,
        }
    return {'success': False, 'error': res.message,
            'charge': np.zeros(T), 'discharge': np.zeros(T),
            'soc': np.full(T, soc_start_kwh), 'grid_import': base_load.clip(0),
            'grid_export': (-base_load).clip(0), 'soc_eind': soc_start_kwh}

### 5a. Optimalisatie over alle dagen

De optimalisatie loopt dag per dag. De eindstand van de batterij aan het
einde van dag *d* is de beginstand van dag *d+1*.

In [ ]:
# Koopprijzen en verkoopprijzen per kwartier voor dynamisch tarief
df['koop_dyn']   = df['epex_eur_mwh'].fillna(df['epex_eur_mwh'].median()) / 1000 + DYNAMISCH['opslag_kwh']
df['verkoop_dyn'] = df['epex_eur_mwh'].fillna(df['epex_eur_mwh'].median()) / 1000 - DYNAMISCH['injectie_korting']

# Arrays voor geoptimaliseerde flows
n = len(df)
charge_opt    = np.zeros(n)
discharge_opt = np.zeros(n)
soc_opt       = np.zeros(n)
import_opt    = np.zeros(n)
export_opt    = np.zeros(n)

soc_now = BAT['soc_start'] * BAT['cap_kwh']
datum_array = df['datum'].values
dagen = sorted(set(datum_array))

print(f'Optimaliseren over {len(dagen)} dagen...')
mislukt = 0
for dag in dagen:
    idx = np.where(datum_array == dag)[0]
    if len(idx) < 96:
        # Onvolledige dag: overslaan, geen batterijdispatch
        import_opt[idx]  = df['base_load_kwh'].iloc[idx].clip(lower=0).values
        export_opt[idx]  = (-df['base_load_kwh'].iloc[idx]).clip(lower=0).values
        soc_opt[idx]     = soc_now
        continue
    bl    = df['base_load_kwh'].iloc[idx].values
    koop  = df['koop_dyn'].iloc[idx].values
    verk  = df['verkoop_dyn'].iloc[idx].values
    res   = optimize_bat_dag(bl, koop, verk, soc_now)
    charge_opt[idx]    = res['charge']
    discharge_opt[idx] = res['discharge']
    soc_opt[idx]       = res['soc']
    import_opt[idx]    = res['grid_import']
    export_opt[idx]    = res['grid_export']
    soc_now = res['soc_eind']
    if not res['success']:
        mislukt += 1

df['import_opt']    = import_opt
df['export_opt']    = export_opt
df['charge_opt']    = charge_opt
df['discharge_opt'] = discharge_opt
df['soc_opt']       = soc_opt

print(f'Klaar. Mislukte optimalisaties: {mislukt}')
print(f'Totale afname optimaal : {import_opt.sum():.1f} kWh')
print(f'Totale injectie optimaal: {export_opt.sum():.1f} kWh')

In [ ]:
# Kosten scenario B: geoptimaliseerde batterij + dynamisch tarief
df['ek_dyn_opt'] = energiekosten_dynamisch(
    df['import_opt'], df['export_opt'], df['epex_eur_mwh'].fillna(df['epex_eur_mwh'].median())
)

# Variabele netkosten met geoptimaliseerde flows
# (capaciteitstarief berekend op basis van maandpiek import_opt)
df['nk_kwartier_opt'] = netkosten_kwartier(df.assign(
    afname_kwh=df['import_opt'], injectie_kwh=df['export_opt']
))

cap_opt = capaciteitstarief_maand(pd.Series(import_opt, index=df.index), df['kwartier'])

# Maandoverzicht alle scenario's
cap_nb.index  = cap_nb.index.to_period('M')
cap_opt.index = cap_opt.index.to_period('M')

maand_b = df.groupby('maand').agg(
    ek_dyn_opt  = ('ek_dyn_opt',      'sum'),
    nk_opt_var  = ('nk_kwartier_opt', 'sum'),
).round(2)
maand_b = maand_b.join(cap_opt[['piek_kw','cap_kost']].rename(
    columns={'piek_kw':'piek_opt','cap_kost':'cap_opt'}), how='left')
maand_b['cap_opt'] = maand_b['cap_opt'].fillna(0)

btw = 1 + NETKOSTEN['btw']
maand_b['totaal_dyn_opt'] = (
    (maand_b['ek_dyn_opt'] + DYNAMISCH['maandkost']) * (1 + DYNAMISCH['btw'])
    + (maand_b['nk_opt_var'] + maand_b['cap_opt'] + NETKOSTEN['meter_mnd'] + NETKOSTEN['heffingen_mnd']) * btw
)

# Combineer met scenario A
vergelijk = maand_a[['totaal_vast','totaal_dyn']].join(maand_b[['totaal_dyn_opt']])
vergelijk.columns = ['Vast (DATS24)', 'Dynamisch (geen bat)', 'Dynamisch (bat opt)']
vergelijk['Besparing bat'] = vergelijk['Dynamisch (geen bat)'] - vergelijk['Dynamisch (bat opt)']
vergelijk['Besparing dyn vs vast'] = vergelijk['Dynamisch (geen bat)'] - vergelijk['Vast (DATS24)']

print('\nMAANDVERGELIJKING (incl. netkosten + BTW)\n')
kop2 = f'{"Maand":<9} {"Vast":>8} {"Dyn":>8} {"Dyn+Bat":>9} {"Bat besparing":>14} {"Dyn-Vast":>10}'
print(kop2)
print('-' * 64)
for m, r in vergelijk.iterrows():
    print(f'{str(m):<9}'
          f' {r["Vast (DATS24)"]:>8.2f}'
          f' {r["Dynamisch (geen bat)"]:>8.2f}'
          f' {r["Dynamisch (bat opt)"]:>9.2f}'
          f' {r["Besparing bat"]:>+14.2f}'
          f' {r["Besparing dyn vs vast"]:>+10.2f}')
print('-' * 64)
print(f'{"TOTAAL":<9}'
      f' {vergelijk["Vast (DATS24)"].sum():>8.2f}'
      f' {vergelijk["Dynamisch (geen bat)"].sum():>8.2f}'
      f' {vergelijk["Dynamisch (bat opt)"].sum():>9.2f}'
      f' {vergelijk["Besparing bat"].sum():>+14.2f}'
      f' {vergelijk["Besparing dyn vs vast"].sum():>+10.2f}')

In [ ]:
fig_b = go.Figure()
mx = [str(m) for m in vergelijk.index]

for naam, kleur in [
    ('Vast (DATS24)', 'steelblue'),
    ('Dynamisch (geen bat)', 'darkorange'),
    ('Dynamisch (bat opt)', 'seagreen'),
]:
    fig_b.add_trace(go.Bar(name=naam, x=mx, y=vergelijk[naam], marker_color=kleur))

fig_b.update_layout(
    title='Maandelijkse elektriciteitskost: alle scenario\'s vergelijking',
    yaxis_title='Kost incl. BTW (€/maand)',
    barmode='group',
    legend=dict(orientation='h', y=1.1),
    height=480,
)
fig_b.show()

## 6. Interactieve dagplot

Kies een datum om het uurprofiel te bekijken:
- **EPEX-prijs per uur** (rechter as)
- **Netbelasting** (basisbelasting zonder batterij vs geoptimaliseerd met batterij)
- **Batterij SOC** bij geoptimaliseerde dispatch

In [ ]:
from datetime import date as date_type

# Voeg datum-kolom als date-type toe (voor de widget)
alle_datums = sorted(set(df['datum'].values))
midden_datum = alle_datums[len(alle_datums) // 2]

datum_picker = widgets.DatePicker(
    description='Datum:',
    value=midden_datum,
    min=alle_datums[0],
    max=alle_datums[-1],
    disabled=False,
    layout=widgets.Layout(width='250px'),
)
output = widgets.Output()

def toon_dag(dag):
    sel = df[df['datum'] == dag].copy()
    if sel.empty:
        print(f'Geen data voor {dag}')
        return
    tijdlabels = [pd.Timestamp(sel['kwartier'].iloc[i]).strftime('%H:%M') for i in range(len(sel))]
    epex_prijs = sel['epex_eur_mwh'].fillna(0).values

    fig_dag = make_subplots(
        rows=3, cols=1,
        shared_xaxes=True,
        subplot_titles=[
            'EPEX prijs (dag-vooruit)',
            'Netbelasting per kwartier (kWh)',
            'Batterij SOC (geoptimaliseerd)',
        ],
        vertical_spacing=0.07,
        row_heights=[0.25, 0.45, 0.30],
    )

    # Rij 1: EPEX prijs
    fig_dag.add_trace(go.Bar(
        x=tijdlabels, y=epex_prijs,
        name='EPEX €/MWh',
        marker_color=np.where(epex_prijs < 0, 'crimson', 'goldenrod'),
        showlegend=True,
    ), row=1, col=1)

    # Rij 2: netbelasting
    fig_dag.add_trace(go.Bar(
        x=tijdlabels, y=sel['afname_nb'].values,
        name='Afname (geen bat)',
        marker_color='steelblue', opacity=0.6,
    ), row=2, col=1)
    fig_dag.add_trace(go.Bar(
        x=tijdlabels, y=-sel['injectie_nb'].values,
        name='Injectie (geen bat)',
        marker_color='darkorange', opacity=0.6,
    ), row=2, col=1)
    fig_dag.add_trace(go.Scatter(
        x=tijdlabels, y=sel['import_opt'].values,
        name='Afname (bat opt)',
        mode='lines', line=dict(color='seagreen', width=2),
    ), row=2, col=1)
    fig_dag.add_trace(go.Scatter(
        x=tijdlabels, y=-sel['export_opt'].values,
        name='Injectie (bat opt)',
        mode='lines', line=dict(color='salmon', width=2),
    ), row=2, col=1)

    # Rij 3: SOC
    soc_pct = sel['soc_opt'].values / BAT['cap_kwh'] * 100
    fig_dag.add_trace(go.Scatter(
        x=tijdlabels, y=soc_pct,
        name='SOC %', mode='lines+markers',
        line=dict(color='purple', width=2),
        fill='tozeroy', fillcolor='rgba(128,0,128,0.15)',
    ), row=3, col=1)
    fig_dag.add_hline(y=BAT['soc_max']*100, line_dash='dot',
                      line_color='gray', row=3, col=1)
    fig_dag.add_hline(y=BAT['soc_min']*100, line_dash='dot',
                      line_color='gray', row=3, col=1)

    # Kostlabels
    if dag in maand_a.index.map(lambda m: m.to_timestamp().date()).values:
        pass  # maandwaarden, geen dag-uitsplitsing hier
    dag_sel = df[df['datum'] == dag]
    kost_vast = (dag_sel['ek_vast_nb'] + dag_sel['nk_kwartier']).sum() * (1 + DATS24['btw'])
    kost_dyn  = (dag_sel['ek_dyn_nb']  + dag_sel['nk_kwartier']).sum() * (1 + DYNAMISCH['btw'])
    kost_opt  = (dag_sel['ek_dyn_opt'] + dag_sel['nk_kwartier_opt']).sum() * (1 + DYNAMISCH['btw'])

    fig_dag.update_layout(
        title=f'Dagprofiel {dag}  |  Vast: €{kost_vast:.2f}  Dyn: €{kost_dyn:.2f}  Bat-opt: €{kost_opt:.2f}',
        height=650,
        legend=dict(orientation='h', y=-0.05),
        yaxis_title='€/MWh',
        yaxis2_title='kWh',
        yaxis3_title='SOC %',
        xaxis3_title='Tijdstip',
    )
    fig_dag.show()

def on_datum_change(change):
    with output:
        output.clear_output(wait=True)
        toon_dag(change['new'])

datum_picker.observe(on_datum_change, names='value')

display(datum_picker, output)
with output:
    toon_dag(datum_picker.value)

## 7. Samenvatting

| Scenario | Jaarlijkse kost (schatting) | t.o.v. vast tarief |
|---|---|---|
| Vast DATS24 (geen bat) | zie tabel | — |
| Dynamisch EPEX (geen bat) | zie tabel | zie `Besparing dyn vs vast` |
| Dynamisch EPEX + bat opt | zie tabel | zie `Besparing bat` |

**Methodologische beperkingen:**

- Het capaciteitstarief bij LP-optimalisatie minimaliseert **energiekosten**;
  piekbegrenzing is niet in de doelfunctie opgenomen.
  Voor volledig piekmijden: voeg een maandpiek-constraint toe aan het LP.
- EV-laden is als vaste belasting beschouwd (niet geoptimaliseerd).
- Solaire productie is impliciet verwerkt via de reconstructie van de basisbelasting.
- DATS24-tariefwaarden zijn april-2026 schattingen; je eigen contract kan afwijken.

In [ ]:
print('Totaaloverzicht (€ incl. BTW, over volledige beschikbare periode):')
print(f'  Vast DATS24 (geen bat)     : {vergelijk["Vast (DATS24)"].sum():>8.2f} €')
print(f'  Dynamisch EPEX (geen bat)  : {vergelijk["Dynamisch (geen bat)"].sum():>8.2f} €')
print(f'  Dynamisch EPEX (bat opt)   : {vergelijk["Dynamisch (bat opt)"].sum():>8.2f} €')
print()
dyn_vs_vast = vergelijk['Dynamisch (geen bat)'].sum() - vergelijk['Vast (DATS24)'].sum()
bat_besparing = vergelijk['Besparing bat'].sum()
print(f'  Dynamisch vs vast          : {dyn_vs_vast:>+8.2f} € ({dyn_vs_vast/vergelijk["Vast (DATS24)"].sum()*100:+.1f}%)')
print(f'  Batterijbesparing (dyn)    : {bat_besparing:>+8.2f} € ({bat_besparing/vergelijk["Dynamisch (geen bat)"].sum()*100:+.1f}%)')